# SEC 10종목 재무 데이터 가용성 검증

## tl;dr

매출·순이익·영업현금흐름·자기자본은 10/10에서 최근 8분기를 복원했다. 총부채와 매출총이익은 각각 7/10이며, 6개 필드가 모두 기계적으로 확보된 종목은 5/10이다. SEC Companyfacts만으로는 부족하고 custom inline XBRL fallback과 CIK 계보가 필요하다.

## Context & Methods

- 기준일: 2026-07-29
- 표본: AAPL, MSFT, GOOGL, HD, JNJ, PG, COST, CAT, XOM, LIN
- 분기 손익·현금흐름: 직접 3개월 값 또는 YTD 차이
- 재무상태표: 각 분기 말 기준 최신 접수 사실
- 결측값은 0으로 채우지 않음
- 상세 provenance: `results.json`
- 판정 보고서: `report.md`

In [ ]:
from pathlib import Path
import json
import pandas as pd

base_dir = Path.cwd()
if base_dir.name != "quality-oversold-sec-feasibility":
    base_dir = base_dir / "analysis" / "quality-oversold-sec-feasibility"

payload = json.loads((base_dir / "results.json").read_text(encoding="utf-8"))
payload["as_of"], len(payload["companies"])

## Coverage

In [ ]:
fields = ["revenue", "net_income", "operating_cash_flow", "equity", "total_debt", "gross_profit"]
coverage = pd.DataFrame([
    {
        "symbol": company["symbol"],
        **{field: company["fields"][field]["quarters_available"] for field in fields},
        "all_six": company["all_six_fields_have_8_quarters"],
    }
    for company in payload["companies"]
])
coverage

In [ ]:
pd.DataFrame([
    {
        "field": field,
        "companies_with_8q": payload["companies_with_8_quarters_by_field"][field],
        "tags_used": len(payload["tag_variation"][field]),
    }
    for field in fields
])

## Provenance & exceptions

In [ ]:
provenance = pd.DataFrame([
    {
        "symbol": company["symbol"],
        "facts": company["provenance"]["source_fact_count"],
        "filed": company["provenance"]["filed_present"],
        "accession": company["provenance"]["accession_present"],
        "amended_sources": company["provenance"]["amended_source_fact_count"],
        "financial_cik": company["financial_cik"],
        "lineage_used": company["cik_lineage_used"],
    }
    for company in payload["companies"]
])
provenance

In [ ]:
debt_crosscheck = pd.read_csv(base_dir / "debt_crosscheck.csv")
debt_crosscheck[["symbol", "sec_provisional_de", "yfinance_de", "relative_gap_pct", "sec_debt_quarters"]]

## Takeaways

1. 선택 원천 사실 692개는 filed date와 accession number가 모두 존재한다.
2. GOOGL은 최근 8분기 안에서 매출 표준 태그가 바뀌었다.
3. CAT·XOM·LIN은 동일한 회계 의미의 매출총이익을 표준 사실만으로 만들 수 없다.
4. XOM은 2026년 7월 새 CIK로 승계되어 선행 CIK 연결 없이는 재무 이력이 사라진다.
5. 총부채 포함 범위를 고정하고 원문 inline XBRL fallback을 검증한 뒤 품질 게이트 구현으로 넘어가야 한다.